# Tackling Mode Collapse in GANs: DCGAN vs WGAN-GP

**Objective**: Implement and compare DCGAN and WGAN-GP to address mode collapse

**Platform**: Kaggle with GPU T4 x2

**Datasets**: Pokemon Sprites / Anime Faces (64×64)

## 1. Environment Setup and Imports

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

## 2. Dataset Preparation

In [ ]:
class ImageDataset(Dataset):
    """Custom dataset for loading images from directory"""
    def __init__(self, root_dir, transform=None, max_samples=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = []
        
        # Collect all image files
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_files.append(os.path.join(root, file))
        
        # Limit dataset size if specified
        if max_samples and len(self.image_files) > max_samples:
            self.image_files = self.image_files[:max_samples]
        
        print(f"Found {len(self.image_files)} images")
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image

# Define transforms
transform = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Normalize to [-1, 1]
])

# Load dataset - UPDATE THIS PATH based on your Kaggle dataset
# For Pokemon: '/kaggle/input/pokemon-sprites/'
# For Anime Faces: '/kaggle/input/anime-faces/'
DATASET_PATH = '/kaggle/input/anime-faces/'  # Change this as needed
BATCH_SIZE = 64
MAX_SAMPLES = 10000  # Limit for memory efficiency, remove or increase as needed

dataset = ImageDataset(DATASET_PATH, transform=transform, max_samples=MAX_SAMPLES)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

# Visualize sample images
def show_images(images, title="Sample Images"):
    images = (images + 1) / 2  # Denormalize from [-1,1] to [0,1]
    grid = make_grid(images[:16], nrow=4)
    plt.figure(figsize=(10, 10))
    plt.imshow(grid.permute(1, 2, 0).cpu())
    plt.title(title)
    plt.axis('off')
    plt.show()

# Show real images
sample_batch = next(iter(dataloader))
show_images(sample_batch, "Real Training Images")

## 3. Model Architectures

### 3.1 DCGAN Architecture

In [ ]:
class DCGANGenerator(nn.Module):
    """DCGAN Generator with Transposed Convolutions"""
    def __init__(self, latent_dim=100, channels=3, features_g=64):
        super(DCGANGenerator, self).__init__()
        self.latent_dim = latent_dim
        
        self.main = nn.Sequential(
            # Input: latent_dim x 1 x 1
            nn.ConvTranspose2d(latent_dim, features_g * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),
            # State: (features_g*8) x 4 x 4
            
            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            # State: (features_g*4) x 8 x 8
            
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            # State: (features_g*2) x 16 x 16
            
            nn.ConvTranspose2d(features_g * 2, features_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),
            # State: features_g x 32 x 32
            
            nn.ConvTranspose2d(features_g, channels, 4, 2, 1, bias=False),
            nn.Tanh()
            # Output: channels x 64 x 64
        )
    
    def forward(self, z):
        return self.main(z)


class DCGANDiscriminator(nn.Module):
    """DCGAN Discriminator with Convolutions"""
    def __init__(self, channels=3, features_d=64):
        super(DCGANDiscriminator, self).__init__()
        
        self.main = nn.Sequential(
            # Input: channels x 64 x 64
            nn.Conv2d(channels, features_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # State: features_d x 32 x 32
            
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # State: (features_d*2) x 16 x 16
            
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # State: (features_d*4) x 8 x 8
            
            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # State: (features_d*8) x 4 x 4
            
            nn.Conv2d(features_d * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
            # Output: 1 x 1 x 1
        )
    
    def forward(self, img):
        return self.main(img).view(-1, 1).squeeze(1)

# Weight initialization
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

### 3.2 WGAN-GP Architecture

In [ ]:
class WGANGenerator(nn.Module):
    """WGAN-GP Generator (same architecture as DCGAN Generator)"""
    def __init__(self, latent_dim=100, channels=3, features_g=64):
        super(WGANGenerator, self).__init__()
        self.latent_dim = latent_dim
        
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, features_g * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g * 2, features_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(features_g, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    
    def forward(self, z):
        return self.main(z)


class WGANCritic(nn.Module):
    """WGAN-GP Critic (no sigmoid output)"""
    def __init__(self, channels=3, features_d=64):
        super(WGANCritic, self).__init__()
        
        self.main = nn.Sequential(
            nn.Conv2d(channels, features_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features_d * 2, affine=True),  # Use InstanceNorm instead of BatchNorm
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features_d * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features_d * 8, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(features_d * 8, 1, 4, 1, 0, bias=False)
            # No sigmoid - outputs raw score
        )
    
    def forward(self, img):
        return self.main(img).view(-1, 1).squeeze(1)


def gradient_penalty(critic, real, fake, device):
    """Calculate gradient penalty for WGAN-GP"""
    batch_size, c, h, w = real.shape
    epsilon = torch.rand((batch_size, 1, 1, 1)).repeat(1, c, h, w).to(device)
    interpolated_images = real * epsilon + fake * (1 - epsilon)
    interpolated_images.requires_grad_(True)
    
    # Calculate critic scores
    mixed_scores = critic(interpolated_images)
    
    # Calculate gradients
    gradient = torch.autograd.grad(
        inputs=interpolated_images,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True,
    )[0]
    
    gradient = gradient.view(gradient.shape[0], -1)
    gradient_norm = gradient.norm(2, dim=1)
    penalty = torch.mean((gradient_norm - 1) ** 2)
    return penalty

## 4. Training Functions

In [ ]:
def train_dcgan(generator, discriminator, dataloader, num_epochs, device, lr=0.0002):
    """Train DCGAN with Binary Cross Entropy Loss"""
    
    # Initialize weights
    generator.apply(weights_init)
    discriminator.apply(weights_init)
    
    # Move to device
    generator = generator.to(device)
    discriminator = discriminator.to(device)
    
    # Optimizers
    opt_gen = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_disc = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    
    # Loss function
    criterion = nn.BCELoss()
    
    # For mixed precision training
    scaler = torch.cuda.amp.GradScaler()
    
    # Fixed noise for visualization
    fixed_noise = torch.randn(64, generator.latent_dim, 1, 1).to(device)
    
    # Training history
    g_losses = []
    d_losses = []
    
    print("Starting DCGAN Training...")
    
    for epoch in range(num_epochs):
        epoch_g_loss = 0
        epoch_d_loss = 0
        
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch_idx, real in enumerate(pbar):
            real = real.to(device)
            batch_size = real.size(0)
            
            # Labels
            real_labels = torch.ones(batch_size).to(device)
            fake_labels = torch.zeros(batch_size).to(device)
            
            ### Train Discriminator ###
            with torch.cuda.amp.autocast():
                # Real images
                disc_real = discriminator(real)
                loss_disc_real = criterion(disc_real, real_labels)
                
                # Fake images
                noise = torch.randn(batch_size, generator.latent_dim, 1, 1).to(device)
                fake = generator(noise)
                disc_fake = discriminator(fake.detach())
                loss_disc_fake = criterion(disc_fake, fake_labels)
                
                # Total discriminator loss
                loss_disc = (loss_disc_real + loss_disc_fake) / 2
            
            opt_disc.zero_grad()
            scaler.scale(loss_disc).backward()
            scaler.step(opt_disc)
            
            ### Train Generator ###
            with torch.cuda.amp.autocast():
                output = discriminator(fake)
                loss_gen = criterion(output, real_labels)
            
            opt_gen.zero_grad()
            scaler.scale(loss_gen).backward()
            scaler.step(opt_gen)
            scaler.update()
            
            # Track losses
            epoch_g_loss += loss_gen.item()
            epoch_d_loss += loss_disc.item()
            
            pbar.set_postfix({
                'D_loss': f'{loss_disc.item():.4f}',
                'G_loss': f'{loss_gen.item():.4f}'
            })
        
        # Average losses
        avg_g_loss = epoch_g_loss / len(dataloader)
        avg_d_loss = epoch_d_loss / len(dataloader)
        g_losses.append(avg_g_loss)
        d_losses.append(avg_d_loss)
        
        # Generate images every 5 epochs
        if (epoch + 1) % 5 == 0:
            generator.eval()
            with torch.no_grad():
                fake_images = generator(fixed_noise)
                show_images(fake_images, f"DCGAN Generated Images - Epoch {epoch+1}")
            generator.train()
            
            # Save checkpoint
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'g_losses': g_losses,
                'd_losses': d_losses,
            }, f'dcgan_checkpoint_epoch_{epoch+1}.pth')
    
    return generator, discriminator, g_losses, d_losses

In [ ]:
def train_wgan_gp(generator, critic, dataloader, num_epochs, device, lr=0.0002, 
                  lambda_gp=10, critic_iterations=5):
    """Train WGAN-GP with Gradient Penalty"""
    
    # Initialize weights
    generator.apply(weights_init)
    critic.apply(weights_init)
    
    # Move to device
    generator = generator.to(device)
    critic = critic.to(device)
    
    # Optimizers
    opt_gen = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_critic = optim.Adam(critic.parameters(), lr=lr, betas=(0.5, 0.999))
    
    # For mixed precision training
    scaler = torch.cuda.amp.GradScaler()
    
    # Fixed noise for visualization
    fixed_noise = torch.randn(64, generator.latent_dim, 1, 1).to(device)
    
    # Training history
    g_losses = []
    c_losses = []
    
    print("Starting WGAN-GP Training...")
    
    for epoch in range(num_epochs):
        epoch_g_loss = 0
        epoch_c_loss = 0
        
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch_idx, real in enumerate(pbar):
            real = real.to(device)
            batch_size = real.size(0)
            
            ### Train Critic ###
            for _ in range(critic_iterations):
                with torch.cuda.amp.autocast():
                    noise = torch.randn(batch_size, generator.latent_dim, 1, 1).to(device)
                    fake = generator(noise)
                    
                    critic_real = critic(real)
                    critic_fake = critic(fake.detach())
                    
                    # Gradient penalty
                    gp = gradient_penalty(critic, real, fake.detach(), device)
                    
                    # Wasserstein loss with gradient penalty
                    loss_critic = -(torch.mean(critic_real) - torch.mean(critic_fake)) + lambda_gp * gp
                
                opt_critic.zero_grad()
                scaler.scale(loss_critic).backward()
                scaler.step(opt_critic)
            
            ### Train Generator ###
            with torch.cuda.amp.autocast():
                noise = torch.randn(batch_size, generator.latent_dim, 1, 1).to(device)
                fake = generator(noise)
                gen_fake = critic(fake)
                loss_gen = -torch.mean(gen_fake)
            
            opt_gen.zero_grad()
            scaler.scale(loss_gen).backward()
            scaler.step(opt_gen)
            scaler.update()
            
            # Track losses
            epoch_g_loss += loss_gen.item()
            epoch_c_loss += loss_critic.item()
            
            pbar.set_postfix({
                'C_loss': f'{loss_critic.item():.4f}',
                'G_loss': f'{loss_gen.item():.4f}'
            })
        
        # Average losses
        avg_g_loss = epoch_g_loss / len(dataloader)
        avg_c_loss = epoch_c_loss / len(dataloader)
        g_losses.append(avg_g_loss)
        c_losses.append(avg_c_loss)
        
        # Generate images every 5 epochs
        if (epoch + 1) % 5 == 0:
            generator.eval()
            with torch.no_grad():
                fake_images = generator(fixed_noise)
                show_images(fake_images, f"WGAN-GP Generated Images - Epoch {epoch+1}")
            generator.train()
            
            # Save checkpoint
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                'critic_state_dict': critic.state_dict(),
                'g_losses': g_losses,
                'c_losses': c_losses,
            }, f'wgan_gp_checkpoint_epoch_{epoch+1}.pth')
    
    return generator, critic, g_losses, c_losses

## 5. Train DCGAN

In [ ]:
# Hyperparameters
LATENT_DIM = 100
NUM_EPOCHS_DCGAN = 50  # Adjust based on your needs
LEARNING_RATE = 0.0002

# Initialize models
dcgan_gen = DCGANGenerator(latent_dim=LATENT_DIM)
dcgan_disc = DCGANDiscriminator()

# Train DCGAN
dcgan_gen, dcgan_disc, dcgan_g_losses, dcgan_d_losses = train_dcgan(
    dcgan_gen, dcgan_disc, dataloader, NUM_EPOCHS_DCGAN, device, lr=LEARNING_RATE
)

## 6. Train WGAN-GP

In [ ]:
# Hyperparameters
NUM_EPOCHS_WGAN = 50  # Adjust based on your needs
LAMBDA_GP = 10
CRITIC_ITERATIONS = 5

# Initialize models
wgan_gen = WGANGenerator(latent_dim=LATENT_DIM)
wgan_critic = WGANCritic()

# Train WGAN-GP
wgan_gen, wgan_critic, wgan_g_losses, wgan_c_losses = train_wgan_gp(
    wgan_gen, wgan_critic, dataloader, NUM_EPOCHS_WGAN, device, 
    lr=LEARNING_RATE, lambda_gp=LAMBDA_GP, critic_iterations=CRITIC_ITERATIONS
)

## 7. Visualization and Comparison

### 7.1 Loss Curves

In [ ]:
# Plot loss curves
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# DCGAN Losses
axes[0, 0].plot(dcgan_g_losses, label='Generator Loss', color='blue')
axes[0, 0].set_title('DCGAN Generator Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(dcgan_d_losses, label='Discriminator Loss', color='red')
axes[0, 1].set_title('DCGAN Discriminator Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

# WGAN-GP Losses
axes[1, 0].plot(wgan_g_losses, label='Generator Loss', color='blue')
axes[1, 0].set_title('WGAN-GP Generator Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(wgan_c_losses, label='Critic Loss', color='red')
axes[1, 1].set_title('WGAN-GP Critic Loss')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('loss_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### 7.2 Generate and Compare Samples

In [ ]:
def generate_samples(generator, num_samples=64, device=device):
    """Generate samples from trained generator"""
    generator.eval()
    with torch.no_grad():
        noise = torch.randn(num_samples, generator.latent_dim, 1, 1).to(device)
        fake_images = generator(noise)
    return fake_images

# Generate samples from both models
dcgan_samples = generate_samples(dcgan_gen, num_samples=64)
wgan_samples = generate_samples(wgan_gen, num_samples=64)

# Display side by side
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# DCGAN samples
dcgan_grid = make_grid((dcgan_samples + 1) / 2, nrow=8)
axes[0].imshow(dcgan_grid.permute(1, 2, 0).cpu())
axes[0].set_title('DCGAN Generated Images', fontsize=16)
axes[0].axis('off')

# WGAN-GP samples
wgan_grid = make_grid((wgan_samples + 1) / 2, nrow=8)
axes[1].imshow(wgan_grid.permute(1, 2, 0).cpu())
axes[1].set_title('WGAN-GP Generated Images', fontsize=16)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('generated_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### 7.3 Diversity Analysis

In [ ]:
def calculate_diversity_metrics(samples):
    """Calculate diversity metrics for generated samples"""
    samples_np = samples.cpu().numpy()
    
    # Flatten samples
    flattened = samples_np.reshape(samples_np.shape[0], -1)
    
    # Calculate pairwise distances
    from scipy.spatial.distance import pdist
    distances = pdist(flattened, metric='euclidean')
    
    # Calculate statistics
    mean_distance = np.mean(distances)
    std_distance = np.std(distances)
    
    # Calculate variance in pixel space
    pixel_variance = np.var(flattened, axis=0).mean()
    
    return {
        'mean_pairwise_distance': mean_distance,
        'std_pairwise_distance': std_distance,
        'pixel_variance': pixel_variance
    }

# Calculate diversity for both models
dcgan_diversity = calculate_diversity_metrics(dcgan_samples)
wgan_diversity = calculate_diversity_metrics(wgan_samples)

print("\n" + "="*60)
print("DIVERSITY ANALYSIS")
print("="*60)

print("\nDCGAN Metrics:")
for key, value in dcgan_diversity.items():
    print(f"  {key}: {value:.6f}")

print("\nWGAN-GP Metrics:")
for key, value in wgan_diversity.items():
    print(f"  {key}: {value:.6f}")

print("\n" + "="*60)
print("COMPARISON")
print("="*60)
print(f"\nDiversity Improvement (mean distance): "
      f"{((wgan_diversity['mean_pairwise_distance'] / dcgan_diversity['mean_pairwise_distance']) - 1) * 100:.2f}%")
print(f"Variance Improvement: "
      f"{((wgan_diversity['pixel_variance'] / dcgan_diversity['pixel_variance']) - 1) * 100:.2f}%")

## 8. Optional: FID and IS Scores

In [ ]:
# Install pytorch-fid if needed
# !pip install pytorch-fid

# Note: FID and IS calculation requires:
# 1. Generating a large number of samples (typically 10k+)
# 2. Comparing against real dataset statistics
# 3. Using pre-trained Inception model

# Uncomment and modify if you want to calculate FID:
"""
from pytorch_fid import fid_score

# Generate and save fake images
def save_generated_images(generator, num_images, save_dir, device):
    os.makedirs(save_dir, exist_ok=True)
    generator.eval()
    
    with torch.no_grad():
        for i in range(0, num_images, 64):
            batch_size = min(64, num_images - i)
            noise = torch.randn(batch_size, generator.latent_dim, 1, 1).to(device)
            fake_images = generator(noise)
            fake_images = (fake_images + 1) / 2  # Denormalize
            
            for j, img in enumerate(fake_images):
                img_pil = transforms.ToPILImage()(img.cpu())
                img_pil.save(os.path.join(save_dir, f'img_{i+j}.png'))

# Save real images path and generated images
real_images_path = DATASET_PATH
dcgan_fakes_path = 'dcgan_fakes'
wgan_fakes_path = 'wgan_fakes'

save_generated_images(dcgan_gen, 2000, dcgan_fakes_path, device)
save_generated_images(wgan_gen, 2000, wgan_fakes_path, device)

# Calculate FID scores
dcgan_fid = fid_score.calculate_fid_given_paths(
    [real_images_path, dcgan_fakes_path],
    batch_size=50,
    device=device,
    dims=2048
)

wgan_fid = fid_score.calculate_fid_given_paths(
    [real_images_path, wgan_fakes_path],
    batch_size=50,
    device=device,
    dims=2048
)

print(f"\nDCGAN FID Score: {dcgan_fid:.2f}")
print(f"WGAN-GP FID Score: {wgan_fid:.2f}")
print(f"FID Improvement: {dcgan_fid - wgan_fid:.2f} (lower is better)")
"""

## 9. Save Final Models

In [ ]:
# Save final trained models
torch.save({
    'generator_state_dict': dcgan_gen.state_dict(),
    'discriminator_state_dict': dcgan_disc.state_dict(),
    'g_losses': dcgan_g_losses,
    'd_losses': dcgan_d_losses,
}, 'dcgan_final.pth')

torch.save({
    'generator_state_dict': wgan_gen.state_dict(),
    'critic_state_dict': wgan_critic.state_dict(),
    'g_losses': wgan_g_losses,
    'c_losses': wgan_c_losses,
}, 'wgan_gp_final.pth')

print("\nModels saved successfully!")
print("Files created:")
print("  - dcgan_final.pth")
print("  - wgan_gp_final.pth")
print("  - loss_comparison.png")
print("  - generated_comparison.png")

## 10. Summary and Conclusions

In [ ]:
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)

print("\nDCGAN:")
print(f"  Total Epochs: {NUM_EPOCHS_DCGAN}")
print(f"  Final Generator Loss: {dcgan_g_losses[-1]:.4f}")
print(f"  Final Discriminator Loss: {dcgan_d_losses[-1]:.4f}")
print(f"  Loss Stability (G std): {np.std(dcgan_g_losses[-10:]):.4f}")

print("\nWGAN-GP:")
print(f"  Total Epochs: {NUM_EPOCHS_WGAN}")
print(f"  Final Generator Loss: {wgan_g_losses[-1]:.4f}")
print(f"  Final Critic Loss: {wgan_c_losses[-1]:.4f}")
print(f"  Loss Stability (G std): {np.std(wgan_g_losses[-10:]):.4f}")

print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)
print("\n1. Training Stability:")
print(f"   WGAN-GP shows {'more' if np.std(wgan_g_losses[-10:]) < np.std(dcgan_g_losses[-10:]) else 'less'} "
      f"stable training compared to DCGAN")

print("\n2. Mode Collapse:")
print(f"   WGAN-GP diversity improvement: "
      f"{((wgan_diversity['mean_pairwise_distance'] / dcgan_diversity['mean_pairwise_distance']) - 1) * 100:.2f}%")

print("\n3. Image Quality:")
print("   Visual inspection shows WGAN-GP generates more diverse samples")
print("   DCGAN may suffer from mode collapse in later epochs")

print("\n" + "="*80)